# LensIQ: deploy a face-recognition endpoint (InsightFace `buffalo_l`)

Registers a single MLflow PyFunc that wraps `insightface.app.FaceAnalysis("buffalo_l")`
and serves it behind a Databricks Model Serving endpoint. Each request returns,
for every face detected in the frame, a bounding box, a detection score, and a
512-dim ArcFace embedding.

The AppKit server (server/server.ts) layers identity matching on top by
running a pgvector cosine search against the `faces` table in Lakebase.
Known faces (banned / VIP / staff) are uploaded through the app, embedded
by this same endpoint, and stored in postgres - inference time matching
is a single SQL query against the vector index.

Cold-start lifecycle:

- The buffalo_l model pack (~280MB: SCRFD-10G detector + ArcFace
  w600k_r50 embedder + 5-landmark + gender/age models) is downloaded
  AT NOTEBOOK TIME, verified by loading each ONNX with onnxruntime to
  fail fast on corrupt CDN responses, and logged as an MLflow artifact
  alongside the PyFunc.
- On scale-from-zero the PyFunc's `load_context` copies the bundled
  models into `<INSIGHTFACE_HOME>/models/buffalo_l/` so
  `FaceAnalysis(name="buffalo_l")` finds them locally. NO CDN call at
  serving time - matches the "no external services at inference" rule
  the Roboflow detectors follow. This was added after a CDN download
  returned a truncated `w600k_r50.onnx` and crashed the serving
  container with `onnxruntime ... InvalidProtobuf: Protobuf parsing
  failed.`
- Subsequent requests hit the in-process model and stay local. No
  external API key, no per-frame network calls.

Payload (matches AppKit `serving()` invoke):

```json
{"dataframe_records": [{"image": "<b64>"}]}
```

Response per row:

```json
[{"bbox": [x1, y1, x2, y2], "det_score": 0.94, "embedding": [512 floats]}]
```

In [ ]:
dbutils.widgets.text("catalog", "reggie_pierce_aws_catalog")
dbutils.widgets.text("schema", "lens_iq")
dbutils.widgets.text("registered_name", "lensiq_face_recognition")
dbutils.widgets.text("endpoint_name", "lensiq-face-recognition")
# Detection input size for SCRFD. Larger -> better small-face recall,
# slower. 640 is the recommended default and what the model was trained
# on. Bump to 1024 for crowded scenes.
dbutils.widgets.text("det_size", "640")

In [ ]:
%pip install -q mlflow>=2.13 onnxruntime>=1.16 insightface==0.7.3 opencv-python-headless>=4.8
dbutils.library.restartPython()

In [ ]:
import logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
LOG = logging.getLogger("deploy_face_recognition")

CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
REGISTERED_NAME = dbutils.widgets.get("registered_name").strip()
ENDPOINT = dbutils.widgets.get("endpoint_name").strip()
DET_SIZE = int(dbutils.widgets.get("det_size") or 640)

REGISTERED = f"{CATALOG}.{SCHEMA}.{REGISTERED_NAME}"

LOG.info("Deploying face recognition endpoint %s", ENDPOINT)
LOG.info("  registered_model=%s", REGISTERED)
LOG.info("  det_size=%d", DET_SIZE)

## Stage buffalo_l locally

Download the model pack into the driver, retry on partial / corrupted
responses, and verify every ONNX parses with onnxruntime before we
even start the MLflow run. Anything that gets logged into the
artifact has already been validated end-to-end, which is what
keeps the serving container from booting into the protobuf parse
error we'd otherwise hit when InsightFace silently downloads a
truncated `w600k_r50.onnx`.

In [ ]:
import gc
import hashlib
import os
import shutil
import tempfile
import time
import urllib.error
import urllib.request
import zipfile

BUFFALO_L_URL = "https://github.com/deepinsight/insightface/releases/download/v0.7/buffalo_l.zip"
# Files InsightFace ships in the buffalo_l pack. We only USE the two
# in `BUFFALO_L_USED` at serving time (the PyFunc wrapper passes
# `allowed_modules=["detection","recognition"]` to FaceAnalysis to skip
# the unused 1k3d68 3D-landmark, 2d106det 2D-landmark, and genderage
# models). That keeps cold-start RAM under serverless's ~6GB ceiling -
# loading all 5 plus mlflow's input_example inference path was OOMing.
# We still keep the unused files in the staged dir (and the logged
# artifact) so callers can re-enable additional modules later without
# republishing the artifact.
BUFFALO_L_REQUIRED = (
    "1k3d68.onnx",
    "2d106det.onnx",
    "det_10g.onnx",
    "genderage.onnx",
    "w600k_r50.onnx",
)
BUFFALO_L_USED = (
    "det_10g.onnx",
    "w600k_r50.onnx",
)
BUFFALO_L_DIR = os.path.join(tempfile.gettempdir(), "buffalo_l_staging", "buffalo_l")


def _download_with_retries(url: str, dest: str, attempts: int = 4) -> None:
    """Stream `url` to `dest` with retries. Removes partial files on
    failure so the next attempt starts clean (the truncated download
    is exactly what gave us the original protobuf parse error)."""
    last_err: Exception | None = None
    for attempt in range(1, attempts + 1):
        try:
            tmp = f"{dest}.partial"
            LOG.info("Downloading %s (attempt %d/%d) -> %s", url, attempt, attempts, dest)
            with urllib.request.urlopen(url, timeout=120) as resp:
                expected = int(resp.headers.get("Content-Length") or 0)
                sha = hashlib.sha256()
                read = 0
                with open(tmp, "wb") as f:
                    while True:
                        chunk = resp.read(1 << 20)
                        if not chunk:
                            break
                        f.write(chunk)
                        sha.update(chunk)
                        read += len(chunk)
            if expected and read != expected:
                raise IOError(f"Truncated download: got {read} bytes, expected {expected}")
            os.replace(tmp, dest)
            LOG.info("  %d bytes, sha256=%s", read, sha.hexdigest())
            return
        except (urllib.error.URLError, IOError, TimeoutError) as exc:
            last_err = exc
            LOG.warning("  download attempt %d failed: %s", attempt, exc)
            for path in (dest, f"{dest}.partial"):
                try: os.remove(path)
                except OSError: pass
            time.sleep(min(2 ** attempt, 30))
    raise RuntimeError(f"Failed to download {url} after {attempts} attempts: {last_err}")


def _validate_onnx(path: str) -> None:
    """Open the ONNX file with onnxruntime to fail fast on protobuf
    corruption. Doing this in the notebook (where we can retry) is the
    whole point of staging the artifact instead of relying on
    InsightFace's lazy CDN download at serving cold-start.

    Drops the session reference + forces a GC pass before returning so
    sequential calls don't accumulate ~50-200MB sessions in memory.
    Without this the validation step alone consumed ~1GB of headroom
    on serverless, which combined with mlflow's input_example load
    later in this notebook was enough to OOM the kernel."""
    import onnxruntime  # noqa: PLC0415

    sess = onnxruntime.InferenceSession(path, providers=["CPUExecutionProvider"])
    del sess
    gc.collect()


def _stage_buffalo_l(target_dir: str) -> None:
    """Download buffalo_l.zip, unpack into `<target_dir>`, validate
    every required ONNX file parses cleanly. Idempotent - skips if
    every file is already present and valid."""
    if all(os.path.exists(os.path.join(target_dir, name)) for name in BUFFALO_L_REQUIRED):
        try:
            for name in BUFFALO_L_USED:
                _validate_onnx(os.path.join(target_dir, name))
            LOG.info("buffalo_l already staged + valid at %s", target_dir)
            return
        except Exception as exc:
            LOG.info("Existing stage failed validation (%s); re-downloading", exc)
            shutil.rmtree(target_dir, ignore_errors=True)

    os.makedirs(target_dir, exist_ok=True)
    zip_path = os.path.join(os.path.dirname(target_dir), "buffalo_l.zip")
    _download_with_retries(BUFFALO_L_URL, zip_path)

    # InsightFace's GitHub-release buffalo_l.zip is FLAT - ONNX files
    # sit at the zip root, no inner buffalo_l/ directory. Mirrors and
    # older releases sometimes wrap in `buffalo_l/<files>`. Detect both
    # by inspecting namelist and pick the extraction target accordingly
    # so the ONNX files always land at <target_dir>/<name>.onnx where
    # InsightFace's loader expects them.
    LOG.info("Unzipping %s", zip_path)
    with zipfile.ZipFile(zip_path) as zf:
        names = zf.namelist()
        has_buffalo_prefix = any(n.startswith("buffalo_l/") for n in names)
        extract_to = os.path.dirname(target_dir) if has_buffalo_prefix else target_dir
        LOG.info("  namelist[:3]=%s has_buffalo_prefix=%s extract_to=%s", names[:3], has_buffalo_prefix, extract_to)
        zf.extractall(extract_to)
    os.remove(zip_path)

    missing = [n for n in BUFFALO_L_REQUIRED if not os.path.exists(os.path.join(target_dir, n))]
    if missing:
        raise RuntimeError(f"buffalo_l zip missing expected files: {missing}")
    # Only validate the two models actually loaded at serving time
    # (see PyFunc wrapper - allowed_modules=detection+recognition).
    # Skipping the other three saves ~1GB of peak RAM during staging.
    for name in BUFFALO_L_USED:
        path = os.path.join(target_dir, name)
        LOG.info("  validating %s (%d bytes)", name, os.path.getsize(path))
        _validate_onnx(path)
    LOG.info("buffalo_l staged + validated at %s", target_dir)


_stage_buffalo_l(BUFFALO_L_DIR)

## PyFunc wrapper

Sets `INSIGHTFACE_HOME` to a writable temp dir BEFORE importing
`insightface`. The library reads the env var at import time to decide
where to download / cache the model zoo. On serverless serving the
default `~/.insightface/` is not always writable, so we point it at
the same `SPARK_LOCAL_DIRS` / `tempfile.gettempdir()` probe pattern
the Roboflow detector PyFunc uses.

The `FaceAnalysis` pipeline runs both the SCRFD detector and the
ArcFace recognition head in a single `.get(img)` call - returned
`Face` objects already carry the 512-dim `normed_embedding` we want
for pgvector cosine search.

In [ ]:
import base64
import io
import os
import tempfile
import uuid

import mlflow
import mlflow.pyfunc
import pandas as pd
from mlflow.models import infer_signature


def _writable_temp_subdir(name: str) -> str:
    """Return a writable subdirectory matching dbx-models::tmp_dir().

    Probes SPARK_LOCAL_DIRS first (low-latency on serving), falls back
    to tempfile.gettempdir(). Actually writes a probe file so we never
    return a path that looks writable (e.g. /local_disk0) but rejects
    writes from the serving user.
    """
    spark_local_dirs = os.environ.get("SPARK_LOCAL_DIRS")
    spark_local_dir = spark_local_dirs.split(",")[0] if spark_local_dirs else None
    for unique in (False, True):
        suffix = f"_{uuid.uuid4().hex}" if unique else ""
        for base in (spark_local_dir, tempfile.gettempdir()):
            if not base:
                continue
            target = os.path.join(base, f"{name}{suffix}")
            try:
                os.makedirs(target, exist_ok=True)
                probe = os.path.join(target, f".probe_{os.getpid()}")
                with open(probe, "w") as f:
                    f.write("")
                os.remove(probe)
                return target
            except (OSError, PermissionError):
                continue
    raise PermissionError(f"No writable temp dir found for {name}")


class FaceRecognizer(mlflow.pyfunc.PythonModel):
    """PyFunc serving InsightFace buffalo_l for face detection + 512-d
    ArcFace embeddings.

    Inputs (per row):
      - image: base64-encoded JPEG/PNG (with or without `data:` prefix).

    Output (per row): list of
      {
        bbox:       [x1, y1, x2, y2]  (pixel coords in the input image),
        det_score:  float,            (SCRFD confidence),
        embedding:  list[float] (512) (ArcFace L2-normalized so cosine
                                       similarity is just a dot product).
      }
    """

    def load_context(self, context):
        import shutil  # noqa: PLC0415

        # Stage the bundled buffalo_l artifact into the place InsightFace
        # expects (<INSIGHTFACE_HOME>/models/buffalo_l/). Using the
        # artifact means we never hit the InsightFace CDN at serving
        # time, which is what broke the prior deploy (truncated
        # download -> protobuf parse error on cold start).
        home = _writable_temp_subdir("insightface_home")
        os.environ["INSIGHTFACE_HOME"] = home
        os.environ.setdefault("OMP_NUM_THREADS", "4")
        os.environ.setdefault("ONNXRUNTIME_LOG_SEVERITY_LEVEL", "3")

        models_dir = os.path.join(home, "models")
        os.makedirs(models_dir, exist_ok=True)
        target_pack = os.path.join(models_dir, "buffalo_l")
        if not os.path.isdir(target_pack):
            staged = context.artifacts["buffalo_l_dir"]
            shutil.copytree(staged, target_pack)

        from insightface.app import FaceAnalysis  # noqa: PLC0415

        cfg = context.model_config or {}
        det_size = int(cfg.get("det_size") or 640)

        # `name` selects the pack under <home>/models/<name>/, which we
        # just populated from the bundled artifact.
        # `allowed_modules=["detection","recognition"]` loads ONLY
        # SCRFD-10G (det_10g.onnx) + ArcFace w600k_r50 (512-d
        # embedding). The other three files in the pack (1k3d68 3D
        # landmarks, 2d106det 2D landmarks, genderage age/gender) sit
        # on disk but never get into RAM - that's the trim that keeps
        # cold-start under the serverless ~6GB ceiling.
        # CPUExecutionProvider so we don't need onnxruntime-gpu.
        self._app = FaceAnalysis(
            name="buffalo_l",
            allowed_modules=["detection", "recognition"],
            providers=["CPUExecutionProvider"],
        )
        self._app.prepare(ctx_id=0, det_size=(det_size, det_size))

        from PIL import Image  # noqa: PLC0415
        self._Image = Image

    def _decode(self, image_b64):
        if not image_b64:
            return None
        if isinstance(image_b64, str) and image_b64.startswith("data:"):
            image_b64 = image_b64.split(",", 1)[1]
        return self._Image.open(io.BytesIO(base64.b64decode(image_b64))).convert("RGB")

    def _run_one(self, image_b64):
        img = self._decode(image_b64)
        if img is None:
            return []
        import numpy as np  # noqa: PLC0415

        # InsightFace expects BGR (it was trained on opencv-decoded
        # frames), so flip the RGB array we got from PIL.
        rgb = np.array(img)
        bgr = rgb[:, :, ::-1]
        faces = self._app.get(bgr)
        out = []
        for f in faces:
            x1, y1, x2, y2 = (int(round(v)) for v in f.bbox.tolist())
            # normed_embedding is L2-normalized so pgvector's
            # `<=>` cosine distance == 1 - dot(a, b) directly.
            emb = getattr(f, "normed_embedding", None)
            if emb is None:
                continue
            out.append({
                "bbox": [x1, y1, x2, y2],
                "det_score": float(getattr(f, "det_score", 0.0)),
                "embedding": [float(v) for v in emb.tolist()],
            })
        return out

    def predict(self, context, model_input, params=None):
        if hasattr(model_input, "to_dict"):
            rows = model_input.to_dict(orient="records")
        elif isinstance(model_input, dict):
            rows = [model_input]
        else:
            rows = list(model_input)
        return [self._run_one(r.get("image")) for r in rows]

## Log + register

In [ ]:
_TINY_PNG_B64 = (
    "iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAYAAAAfFcSJAAAADUlEQVR42mP8/5+hHgAH"
    "ggJ/PchI7wAAAABJRU5ErkJggg=="
)

sample_input = pd.DataFrame([
    {"image": _TINY_PNG_B64},
])
sample_output = [[]]
signature = infer_signature(sample_input, sample_output)

mlflow.set_registry_uri("databricks-uc")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")

model_config = {
    "det_size": DET_SIZE,
}

with mlflow.start_run(run_name="deploy_face_recognition") as run:
    info = mlflow.pyfunc.log_model(
        artifact_path="model",
        python_model=FaceRecognizer(),
        signature=signature,
        # input_example is intentionally OMITTED. When set, mlflow spawns
        # a subprocess to verify the example deserializes + the model
        # loads, which means InsightFace + onnxruntime get loaded TWICE
        # at log time (parent + validator child) and tips the serverless
        # notebook over the ~6GB RAM ceiling -> "Execution ran out of
        # memory". The explicit `signature` above is enough for serving
        # to know the input/output schema.
        registered_model_name=REGISTERED,
        model_config=model_config,
        # buffalo_l ONNX pack (~280MB) staged + validated above. Shipping
        # it inside the artifact removes the cold-start CDN call that
        # broke the prior deploy.
        artifacts={"buffalo_l_dir": BUFFALO_L_DIR},
        pip_requirements=[
            "mlflow>=2.13",
            # InsightFace ships its own model-zoo helpers; onnxruntime is
            # the inference backend; opencv-python-headless avoids the
            # X11 deps the regular opencv-python wheel pulls in.
            "insightface==0.7.3",
            "onnxruntime>=1.16,<2.0",
            "opencv-python-headless>=4.8",
        ],
    )
    mlflow.set_tag("lensiq.detector_slug", "face_recognition")
    mlflow.set_tag("lensiq.display_name", "Face Recognition")
    mlflow.set_tag("lensiq.model_pack", "buffalo_l")
    mlflow.set_tag("lensiq.det_size", str(DET_SIZE))
LOG.info("Logged model URI: %s", info.model_uri)

## Create / update the serving endpoint

Single endpoint, scale-to-zero, no external secrets needed. First-time
creation builds a container image with insightface + onnxruntime +
opencv-headless (~600MB image) which takes ~10-15 minutes. Subsequent
config updates redeploy the existing image and finish in 1-2 minutes.

Workload size is `Medium` because the SCRFD detector + ArcFace embedder
do real conv work (~80-200 ms per frame on Small CPU); Medium keeps
latency tolerable when 2-4 faces show up in a single frame.

In [ ]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import EndpointCoreConfigInput, ServedEntityInput

client = mlflow.MlflowClient()
versions = client.search_model_versions(f"name='{REGISTERED}'")
latest_version = max(versions, key=lambda v: int(v.version)).version
LOG.info("Deploying %s version %s -> endpoint %s", REGISTERED, latest_version, ENDPOINT)

served = ServedEntityInput(
    entity_name=REGISTERED,
    entity_version=latest_version,
    workload_size="Small",
    scale_to_zero_enabled=True,
    environment_vars={
        # Quiet onnxruntime warning floods that confuse the serving logs.
        "ONNXRUNTIME_LOG_SEVERITY_LEVEL": "3",
    },
)

w = WorkspaceClient()
try:
    w.serving_endpoints.get(name=ENDPOINT)
    LOG.info("Endpoint exists; updating config")
    w.serving_endpoints.update_config(name=ENDPOINT, served_entities=[served])
except Exception:
    LOG.info("Endpoint not found; creating")
    w.serving_endpoints.create(
        name=ENDPOINT,
        config=EndpointCoreConfigInput(name=ENDPOINT, served_entities=[served]),
    )
LOG.info("Submitted deployment for %s; watch Serving UI for readiness.", ENDPOINT)